# setM UMAP Density Enrichment Vectors

Extract arm-wise density-enrichment vectors from saved `*_arms_setM.npz` files only. For each species, this reconstructs occupied microstate centers in delay-embedded wavelet-PCA projection space, embeds those centers into 2D UMAP, computes arm density maps, and exports log2 arm/global enrichment values across UMAP bins.

In [1]:
from pathlib import Path
import pickle
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.exceptions import InconsistentVersionWarning

REPO_ROOT = Path('/Users/meganbishop/slowmodeevo')
if not (REPO_ROOT / 'pooled_user_pipeline.py').exists():
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((path for path in candidates if (path / 'pooled_user_pipeline.py').exists()), REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pipeline as pp

TUNING_REPRESENTATION = 'egocentered_and_normalized_distances'
TUNING_ROOT = REPO_ROOT / 'outputs' / 'single_species_distance_comparison' / 'tuning' / TUNING_REPRESENTATION
OUTPUT_DIR = TUNING_ROOT / 'setM_umap_density_enrichment_vectors'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 14
MICROSTATE_CENTER_CHUNK_ROWS = 100_000
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.15
DENSITY_BINS = 160
DENSITY_SMOOTH_SIGMA = 2.5
DENSITY_WEIGHT = 'occupancy'  # 'occupancy' or 'microstate'
DENSITY_EPS = 1e-9

# Optional manual override. Leave as None to infer from len(projection) - len(states) + 1.
D_EMBED_OVERRIDE = None

setm_files = sorted(TUNING_ROOT.glob('*/*_arms_setM.npz'))
print(f'found {len(setm_files)} setM files')
for path in setm_files:
    print('-', path.parent.name, path.name)

found 8 setM files
- Mus_caroli Mus_caroli_arms_setM.npz
- Mus_musculus Mus_musculus_arms_setM.npz
- Mus_spretus Mus_spretus_arms_setM.npz
- Peromyscus_californicus Peromyscus_californicus_arms_setM.npz
- Peromyscus_gossypinus Peromyscus_gossypinus_arms_setM.npz
- Peromyscus_leucopus Peromyscus_leucopus_arms_setM.npz
- Peromyscus_maniculatus Peromyscus_maniculatus_arms_setM.npz
- Peromyscus_polionotus Peromyscus_polionotus_arms_setM.npz


In [2]:
def resolve_repo_path(path):
    path = Path(path)
    return path if path.is_absolute() else REPO_ROOT / path


def remove_suffix(text, suffix):
    text = str(text)
    return text[:-len(suffix)] if suffix and text.endswith(suffix) else text


def load_projection_result(species_root):
    projection_path = species_root / 'projection_result.pkl'
    if not projection_path.exists():
        raise FileNotFoundError(f'Missing {projection_path}')
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=InconsistentVersionWarning)
        with open(projection_path, 'rb') as handle:
            projection = pickle.load(handle)
    projection['proj_files'] = [str(resolve_repo_path(path)) for path in projection['proj_files']]
    projection['individual_ids'] = [str(value) for value in projection['individual_ids']]
    return projection


def load_ordered_states(species_root, individual_ids):
    state_files = sorted((species_root / 'states').glob('*_states.npy'))
    states_by_id = {
        remove_suffix(path.stem, '_states'): np.load(path, mmap_mode='r')
        for path in state_files
    }
    missing = [individual_id for individual_id in individual_ids if individual_id not in states_by_id]
    if missing:
        raise KeyError(f'Missing state files for {missing[:10]}')
    return [states_by_id[individual_id] for individual_id in individual_ids]


def infer_d_embed(proj_files, states_list):
    if D_EMBED_OVERRIDE is not None:
        return int(D_EMBED_OVERRIDE)
    inferred = []
    for proj_file, states in zip(proj_files, states_list):
        proj = np.load(proj_file, mmap_mode='r')
        inferred.append(int(proj.shape[0] - len(states) + 1))
    inferred = np.asarray(inferred, dtype=int)
    valid = inferred[inferred > 0]
    if valid.size == 0:
        raise ValueError('Could not infer a positive d_embed from projection/state lengths')
    values, counts = np.unique(valid, return_counts=True)
    d_embed = int(values[np.argmax(counts)])
    if np.any(valid != d_embed):
        print(f'  warning: mixed inferred d_embed values {dict(zip(values, counts))}; using mode {d_embed}')
    return d_embed


def reconstruct_microstate_centers(projection, states_list, n_clusters, d_embed):
    first_proj = np.load(projection['proj_files'][0], mmap_mode='r')
    center_dim = int(d_embed) * int(first_proj.shape[1])
    del first_proj

    center_sums = np.zeros((n_clusters, center_dim), dtype=np.float64)
    cluster_counts = np.zeros(n_clusters, dtype=np.int64)

    for individual_id, proj_file, states in zip(projection['individual_ids'], projection['proj_files'], states_list):
        proj = np.load(proj_file, mmap_mode='r')
        states = np.asarray(states, dtype=int)
        n_rows = len(states)
        for start in range(0, n_rows, MICROSTATE_CENTER_CHUNK_ROWS):
            stop = min(start + MICROSTATE_CENTER_CHUNK_ROWS, n_rows)
            block = pp.delay_embed(proj[start:stop + d_embed - 1], d=d_embed, tau=1)
            block_states = states[start:stop]
            valid = (block_states >= 0) & (block_states < n_clusters)
            if not np.any(valid):
                continue
            np.add.at(center_sums, block_states[valid], block[valid])
            np.add.at(cluster_counts, block_states[valid], 1)
        del proj

    centers = np.divide(
        center_sums,
        np.maximum(cluster_counts[:, None], 1),
        out=np.zeros_like(center_sums),
        where=cluster_counts[:, None] > 0,
    ).astype(np.float32, copy=False)
    return centers, cluster_counts


def embed_microstate_centers(centers, visited, species_root):
    try:
        import os
        os.environ.setdefault('NUMBA_CACHE_DIR', str(species_root / '.numba_cache'))
        import umap
        reducer = umap.UMAP(
            n_neighbors=min(UMAP_N_NEIGHBORS, max(2, int(visited.sum()) - 1)),
            min_dist=UMAP_MIN_DIST,
            metric='euclidean',
            random_state=SEED,
        )
        return reducer.fit_transform(centers[visited]), 'UMAP'
    except Exception as exc:
        xy = PCA(n_components=2, random_state=SEED).fit_transform(centers[visited])
        return xy, f'PCA fallback ({type(exc).__name__})'


def compute_density_enrichment(xy, visited_arms, visited_counts, n_arms):
    x_pad = 0.04 * max(np.ptp(xy[:, 0]), 1e-6)
    y_pad = 0.04 * max(np.ptp(xy[:, 1]), 1e-6)
    x_edges = np.linspace(xy[:, 0].min() - x_pad, xy[:, 0].max() + x_pad, DENSITY_BINS + 1)
    y_edges = np.linspace(xy[:, 1].min() - y_pad, xy[:, 1].max() + y_pad, DENSITY_BINS + 1)

    if DENSITY_WEIGHT == 'occupancy':
        weights = visited_counts.astype(float)
    elif DENSITY_WEIGHT == 'microstate':
        weights = np.ones(len(xy), dtype=float)
    else:
        raise ValueError("DENSITY_WEIGHT must be 'occupancy' or 'microstate'")

    def density(mask):
        hist, _, _ = np.histogram2d(
            xy[mask, 0], xy[mask, 1],
            bins=[x_edges, y_edges],
            weights=weights[mask],
        )
        hist = hist.T
        try:
            from scipy.ndimage import gaussian_filter
            hist = gaussian_filter(hist, sigma=DENSITY_SMOOTH_SIGMA, mode='nearest')
        except Exception:
            pass
        total = hist.sum()
        return hist / total if total > 0 else hist

    global_density = density(np.ones(len(xy), dtype=bool))
    arm_densities = np.stack([density(visited_arms == arm) for arm in range(n_arms)])

    positive_global = global_density[global_density > 0]
    global_floor = np.nanpercentile(positive_global, 10) if positive_global.size else 0
    enrichment = np.log2((arm_densities + DENSITY_EPS) / (global_density[None, :, :] + DENSITY_EPS))
    enrichment = np.where(global_density[None, :, :] > global_floor, enrichment, np.nan)
    return x_edges, y_edges, global_density, arm_densities, enrichment


def export_enrichment_vectors(species, x_edges, y_edges, enrichment, global_density, arm_densities, out_dir):
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    xx, yy = np.meshgrid(x_centers, y_centers)
    n_arms, n_y, n_x = enrichment.shape
    bin_index = np.arange(n_y * n_x)

    wide = pd.DataFrame({
        'species': species,
        'bin_index': bin_index,
        'umap_x': xx.ravel(),
        'umap_y': yy.ravel(),
        'global_density': global_density.ravel(),
    })
    for arm in range(n_arms):
        wide[f'arm_{arm + 1}_density'] = arm_densities[arm].ravel()
        wide[f'arm_{arm + 1}_log2_enrichment'] = enrichment[arm].ravel()

    valid_any = np.any(np.isfinite(enrichment), axis=0).ravel()
    wide_valid = wide.loc[valid_any].reset_index(drop=True)
    wide_path = out_dir / f'{species}_setM_umap_density_enrichment_vectors_wide.csv'
    wide_valid.to_csv(wide_path, index=False)

    long_rows = []
    for arm in range(n_arms):
        values = enrichment[arm].ravel()
        long_rows.append(pd.DataFrame({
            'species': species,
            'arm': arm + 1,
            'bin_index': bin_index,
            'umap_x': xx.ravel(),
            'umap_y': yy.ravel(),
            'global_density': global_density.ravel(),
            'arm_density': arm_densities[arm].ravel(),
            'log2_enrichment': values,
            'finite': np.isfinite(values),
        }))
    long = pd.concat(long_rows, ignore_index=True)
    long = long.loc[long['finite']].reset_index(drop=True)
    long_path = out_dir / f'{species}_setM_umap_density_enrichment_vectors_long.csv'
    long.to_csv(long_path, index=False)
    return wide_path, long_path, wide_valid, long

In [3]:
def process_setm_species(setm_path):
    species_root = setm_path.parent
    species = species_root.name
    out_dir = OUTPUT_DIR / species
    out_dir.mkdir(parents=True, exist_ok=True)

    setm = np.load(setm_path)
    chi = np.asarray(setm['chi'])
    hard_arm = np.asarray(setm['hard_arm'], dtype=int)
    n_clusters, n_arms = chi.shape

    projection = load_projection_result(species_root)
    states_list = load_ordered_states(species_root, projection['individual_ids'])
    d_embed = infer_d_embed(projection['proj_files'], states_list)

    centers, cluster_counts = reconstruct_microstate_centers(
        projection, states_list, n_clusters=n_clusters, d_embed=d_embed
    )
    visited = cluster_counts > 0
    if not np.any(visited):
        raise ValueError(f'{species}: no visited microstates')

    xy, embedding_name = embed_microstate_centers(centers, visited, species_root)
    visited_clusters = np.flatnonzero(visited)
    visited_arms = hard_arm[visited]
    visited_counts = cluster_counts[visited]

    x_edges, y_edges, global_density, arm_densities, enrichment = compute_density_enrichment(
        xy, visited_arms, visited_counts, n_arms
    )

    npz_path = out_dir / f'{species}_setM_umap_density_enrichment_maps.npz'
    np.savez_compressed(
        npz_path,
        xy=xy,
        visited_clusters=visited_clusters,
        visited_arms=visited_arms,
        visited_counts=visited_counts,
        x_edges=x_edges,
        y_edges=y_edges,
        global_density=global_density,
        arm_densities=arm_densities,
        enrichment_maps=enrichment,
    )
    wide_path, long_path, wide, long = export_enrichment_vectors(
        species, x_edges, y_edges, enrichment, global_density, arm_densities, out_dir
    )

    return {
        'species': species,
        'setm_file': str(setm_path),
        'n_clusters': n_clusters,
        'n_arms': n_arms,
        'd_embed': d_embed,
        'center_dim': centers.shape[1],
        'visited_microstates': int(visited.sum()),
        'embedding': embedding_name,
        'density_weight': DENSITY_WEIGHT,
        'density_bins': DENSITY_BINS,
        'npz_path': str(npz_path),
        'wide_csv': str(wide_path),
        'long_csv': str(long_path),
        'wide_rows': len(wide),
        'long_rows': len(long),
    }


inventory_rows = []
for setm_path in setm_files:
    print(f'processing {setm_path.parent.name}')
    inventory_rows.append(process_setm_species(setm_path))

inventory = pd.DataFrame(inventory_rows)
inventory_path = OUTPUT_DIR / 'setM_umap_density_enrichment_inventory.csv'
inventory.to_csv(inventory_path, index=False)
display(inventory)
print('wrote', inventory_path)

processing Mus_caroli


KeyboardInterrupt: 

In [ ]:
# Optional combined exports across species.
wide_frames = []
long_frames = []
for row in inventory.itertuples(index=False):
    wide_frames.append(pd.read_csv(row.wide_csv))
    long_frames.append(pd.read_csv(row.long_csv))

combined_wide = pd.concat(wide_frames, ignore_index=True)
combined_long = pd.concat(long_frames, ignore_index=True)

combined_wide_path = OUTPUT_DIR / 'setM_umap_density_enrichment_vectors_wide_all_species.csv'
combined_long_path = OUTPUT_DIR / 'setM_umap_density_enrichment_vectors_long_all_species.csv'
combined_wide.to_csv(combined_wide_path, index=False)
combined_long.to_csv(combined_long_path, index=False)

print('wrote', combined_wide_path)
print('wrote', combined_long_path)
display(combined_wide.head())
display(combined_long.head())

## Visualize setM UMAP Density Enrichment

Each species has its own UMAP because the saved PCA projections and some delay-embedding dimensions differ across species. These panels are for within-species arm localization and enrichment diagnostics.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

PLOT_DIR = OUTPUT_DIR / 'figures'
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def load_species_maps(row):
    data = np.load(row.npz_path)
    return {key: data[key] for key in data.files}


def plot_species_umap_and_density(row):
    maps = load_species_maps(row)
    species = row.species
    xy = maps['xy']
    arms = maps['visited_arms'].astype(int)
    counts = maps['visited_counts'].astype(float)
    arm_densities = maps['arm_densities']
    enrichment = maps['enrichment_maps']
    x_edges = maps['x_edges']
    y_edges = maps['y_edges']
    extent = [x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]]
    n_arms = arm_densities.shape[0]

    sizes = 8 + 42 * np.sqrt(counts / max(counts.max(), 1))
    colors = plt.cm.tab10(np.linspace(0, 1, max(n_arms, 10)))[:n_arms]

    fig, ax = plt.subplots(figsize=(6.2, 5.4), constrained_layout=True)
    for arm in range(n_arms):
        mask = arms == arm
        if np.any(mask):
            ax.scatter(xy[mask, 0], xy[mask, 1], s=sizes[mask], color=colors[arm], alpha=0.78, linewidths=0, label=f'arm {arm + 1}')
    ax.set_title(f'{species}: setM microstate centers ({row.embedding})')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(frameon=False, fontsize=8)
    scatter_path = PLOT_DIR / f'{species}_setM_microstate_umap_by_arm.png'
    fig.savefig(scatter_path, dpi=200)
    plt.show()

    ncols = min(4, n_arms)
    nrows = int(np.ceil(n_arms / ncols))
    positive = arm_densities[arm_densities > 0]
    vmax = np.nanpercentile(positive, 94) if positive.size else 1
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.5 * nrows), constrained_layout=True, squeeze=False)
    for arm in range(n_arms):
        ax = axes.flat[arm]
        im = ax.imshow(arm_densities[arm], origin='lower', extent=extent, cmap='inferno', norm=PowerNorm(gamma=0.45, vmin=0, vmax=max(vmax, np.finfo(float).eps)), aspect='auto')
        ax.set_title(f'arm {arm + 1} density')
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')
    for ax in axes.flat[n_arms:]:
        ax.set_visible(False)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.78, label=f'{DENSITY_WEIGHT}-weighted density')
    fig.suptitle(f'{species}: setM arm density maps')
    density_path = PLOT_DIR / f'{species}_setM_arm_density_maps.png'
    fig.savefig(density_path, dpi=200)
    plt.show()

    finite_abs = np.abs(enrichment[np.isfinite(enrichment)])
    lim = np.nanpercentile(finite_abs, 92) if finite_abs.size else 1
    lim = max(lim, np.finfo(float).eps)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.5 * nrows), constrained_layout=True, squeeze=False)
    for arm in range(n_arms):
        ax = axes.flat[arm]
        im = ax.imshow(enrichment[arm], origin='lower', extent=extent, cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
        ax.set_title(f'arm {arm + 1} log2 enrichment')
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')
    for ax in axes.flat[n_arms:]:
        ax.set_visible(False)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.78, label='log2 arm/global density')
    fig.suptitle(f'{species}: setM arm density enrichment')
    enrichment_path = PLOT_DIR / f'{species}_setM_arm_density_enrichment_maps.png'
    fig.savefig(enrichment_path, dpi=200)
    plt.show()

    return scatter_path, density_path, enrichment_path


plot_rows = []
for row in inventory.itertuples(index=False):
    paths = plot_species_umap_and_density(row)
    plot_rows.append({'species': row.species, 'scatter_png': str(paths[0]), 'density_png': str(paths[1]), 'enrichment_png': str(paths[2])})

plot_inventory = pd.DataFrame(plot_rows)
plot_inventory_path = OUTPUT_DIR / 'setM_umap_density_enrichment_plot_inventory.csv'
plot_inventory.to_csv(plot_inventory_path, index=False)
display(plot_inventory)
print('wrote', plot_inventory_path)

## JS Divergence and Hierarchical Clustering

Direct JS comparisons between UMAP coordinate bins across species would require a shared cross-species UMAP grid. These saved runs use species-specific PCA bases and mixed delay dimensions, so this section builds coordinate-free representations instead: per-arm histograms of log2 density-enrichment values. The resulting vectors compare how concentrated/selective each arm's density map is, without assuming UMAP coordinates are aligned across species.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, leaves_list
from scipy.spatial.distance import squareform

ENRICHMENT_HIST_BINS = np.linspace(-4, 4, 81)


def normalized_hist(values, bins):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    values = np.clip(values, bins[0], bins[-1])
    hist, _ = np.histogram(values, bins=bins)
    hist = hist.astype(float)
    total = hist.sum()
    if total <= 0:
        return np.full(len(bins) - 1, 1.0 / (len(bins) - 1))
    return hist / total


def js_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    m = 0.5 * (p + q)
    return 0.5 * np.sum(p * np.log2(p / m)) + 0.5 * np.sum(q * np.log2(q / m))


species_vectors = {}
arm_hist_rows = []
for row in inventory.itertuples(index=False):
    maps = load_species_maps(row)
    enrichment = maps['enrichment_maps']
    per_arm = []
    for arm in range(enrichment.shape[0]):
        hist = normalized_hist(enrichment[arm], ENRICHMENT_HIST_BINS)
        per_arm.append(hist)
        arm_hist_rows.append({
            'species': row.species,
            'arm': arm + 1,
            **{f'bin_{i:02d}': value for i, value in enumerate(hist)},
        })
    species_vectors[row.species] = np.concatenate(per_arm)

arm_hist_df = pd.DataFrame(arm_hist_rows)
arm_hist_path = OUTPUT_DIR / 'setM_arm_enrichment_value_histograms.csv'
arm_hist_df.to_csv(arm_hist_path, index=False)

species_names = sorted(species_vectors)
js_matrix = pd.DataFrame(0.0, index=species_names, columns=species_names)
for i, species_i in enumerate(species_names):
    for j, species_j in enumerate(species_names):
        if j <= i:
            continue
        value = js_divergence(species_vectors[species_i], species_vectors[species_j])
        js_matrix.loc[species_i, species_j] = value
        js_matrix.loc[species_j, species_i] = value

js_path = OUTPUT_DIR / 'setM_enrichment_histogram_js_divergence_matrix.csv'
js_matrix.to_csv(js_path)
display(js_matrix)
print('wrote', arm_hist_path)
print('wrote', js_path)

In [ ]:
condensed = squareform(js_matrix.to_numpy(), checks=False)
Z = linkage(condensed, method='average')
order = leaves_list(Z)
ordered_species = [species_names[i] for i in order]
ordered_js = js_matrix.loc[ordered_species, ordered_species]

fig, ax = plt.subplots(figsize=(7.2, 6.4), constrained_layout=True)
im = ax.imshow(ordered_js.to_numpy(), cmap='magma_r')
ax.set_xticks(np.arange(len(ordered_species)))
ax.set_yticks(np.arange(len(ordered_species)))
ax.set_xticklabels(ordered_species, rotation=45, ha='right')
ax.set_yticklabels(ordered_species)
ax.set_title('setM enrichment-histogram JS divergence')
for i in range(len(ordered_species)):
    for j in range(len(ordered_species)):
        ax.text(j, i, f'{ordered_js.iloc[i, j]:.3f}', ha='center', va='center', fontsize=7, color='white' if ordered_js.iloc[i, j] > ordered_js.to_numpy().max() * 0.45 else 'black')
fig.colorbar(im, ax=ax, label='JS divergence')
matrix_png = PLOT_DIR / 'setM_enrichment_histogram_js_divergence_matrix.png'
matrix_pdf = PLOT_DIR / 'setM_enrichment_histogram_js_divergence_matrix.pdf'
fig.savefig(matrix_png, dpi=220)
fig.savefig(matrix_pdf)
plt.show()

fig, ax = plt.subplots(figsize=(7.5, 4.6), constrained_layout=True)
dendrogram(Z, labels=species_names, leaf_rotation=45, ax=ax)
ax.set_title('Hierarchical clustering of setM enrichment histograms')
ax.set_ylabel('average-linkage JS divergence')
dendro_png = PLOT_DIR / 'setM_enrichment_histogram_hierarchical_clustering.png'
dendro_pdf = PLOT_DIR / 'setM_enrichment_histogram_hierarchical_clustering.pdf'
fig.savefig(dendro_png, dpi=220)
fig.savefig(dendro_pdf)
plt.show()

cluster_order = pd.DataFrame({'cluster_order': np.arange(1, len(ordered_species) + 1), 'species': ordered_species})
cluster_order_path = OUTPUT_DIR / 'setM_enrichment_histogram_cluster_order.csv'
cluster_order.to_csv(cluster_order_path, index=False)
display(cluster_order)
print('wrote', matrix_png)
print('wrote', dendro_png)
print('wrote', cluster_order_path)